In [1]:
# Basic Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
df = pd.read_csv("weatherHistory.csv")
print(df.head())
print(df.info())

FileNotFoundError: [Errno 2] No such file or directory: 'weatherHistory.csv'

In [ ]:
# wb to and cos
df['wind_x'] = np.cos(np.radians(df['Wind Bearing (degrees)']))
df['wind_y'] = np.sin(np.radians(df['Wind Bearing (degrees)']))

df = df.drop(['Wind Bearing (degrees)'], axis=1)

In [ ]:

# convert to date time
df['Formatted Date'] = pd.to_datetime(df['Formatted Date'], utc=True)

# Extract 'Month' and 'Year'
df['Month'] = df['Formatted Date'].dt.month
df

In [ ]:
df = df[['Temperature (C)','Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)','Visibility (km)', 'Pressure (millibars)', 'Precip Type','Month','wind_x','wind_y']]
df.columns=['temp','apptem','humidity','wind_speed','vis','pressure','is_rain','Month','wind_x','wind_y']
df

In [ ]:
df = df.dropna(subset=['is_rain'])
df['is_rain'] = df['is_rain'].apply(lambda x: 1 if x.strip().lower() =='rain' else 0)

In [ ]:
df

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features_to_scale = ['temp','apptem', 'humidity', 'wind_speed','vis', 'pressure', 'wind_x', 'wind_y']

# Split first
X = df.drop('is_rain', axis=1)
y = df['is_rain']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Scale  after split
scaler = StandardScaler()
X_train.loc[:, features_to_scale] = scaler.fit_transform(X_train[features_to_scale])
X_test.loc[:, features_to_scale] = scaler.transform(X_test[features_to_scale])

print("Data splitting complete.")
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)


In [ ]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression()

# Train
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Make predictions on the test data
y_pred = model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Print the results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


In [ ]:
#linear regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_test = linear_model.predict(X_test)
y_pred_train = linear_model.predict(X_train)
mse = mean_squared_error(y_test, y_pred_test)

rmse = np.sqrt(mse)

print(f"Test R-squared (R2) Score: {r2_score(y_test, y_pred_test)}")
print(f"Test Mean Squared Error (MSE): {mean_squared_error(y_test, y_pred_test)}")
print(f"Test Root Mean Squared Error (RMSE): {rmse}")


y_pred_test_classified = (y_pred_test > 0.6).astype(int)
print(f"Test Set Accuracy: {accuracy_score(y_test, y_pred_test_classified) * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred_test, alpha=0.5)
plt.plot([0,1],[0,1],'r--')
plt.xlabel("Actual Values (y_test)")
plt.ylabel("Predicted Values (y_pred_test)")
plt.title("Linear Regression: Actual vs Predicted Values")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

y_pred_binary = (y_pred_test > 0.5).astype(int)

# Scatter Plot: Continuous Predictions
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_test, alpha=0.5)
plt.title("Actual vs Predicted (Continuous Output)")
plt.xlabel("Actual is_rain (0 = No Rain, 1 = Rain)")
plt.ylabel("Predicted Value (continuous)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

#  Distribution Plot: Prediction Spread
plt.figure(figsize=(7, 4))
sns.histplot(y_pred_test, bins=30, kde=True, color='royalblue')
plt.title("Distribution of Predicted Continuous Values")
plt.xlabel("Predicted Value")
plt.ylabel("Frequency")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

# Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred_binary)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Rain", "Rain"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix (Binary Classification)")
plt.show()

In [ ]:
features_for_new_model = ['humidity', 'wind_speed', 'pressure', 'vis', 'wind_x', 'wind_y']

X_new = df[features_for_new_model]
y_new = df['is_rain']

from sklearn.model_selection import train_test_split

X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size=0.25, random_state=42)

print("New data splitting complete.")
print("Shape of X_train_new:", X_train_new.shape)
print("Shape of X_test_new:", X_test_new.shape)
print("Shape of y_train_new:", y_train_new.shape)
print("Shape of y_test_new:", y_test_new.shape)



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score
import numpy as np

#  Random Forest
random_forest_new_model = RandomForestClassifier(random_state=42)

# Train
random_forest_new_model.fit(X_train_new, y_train_new)

# predictions on test set
y_pred_rf_new = random_forest_new_model.predict(X_test_new)


rf_new_accuracy = accuracy_score(y_test_new, y_pred_rf_new)
print(f" Random Forest Test Set Accuracy: {rf_new_accuracy * 100:.2f}%")


rf_new_mse = mean_squared_error(y_test_new, y_pred_rf_new)
print(f" Random Forest Test Set Mean Squared Error (MSE): {rf_new_mse:.4f}")

rf_new_rmse = np.sqrt(rf_new_mse)
print(f" Random Forest Test Set Root Mean Squared Error (RMSE): {rf_new_rmse:.4f}")